# Verified three-task portfolio: the public frontier of 2026-11-11, combined and re-verified

This version combines the best public file of each task as of 2026-11-11 21:00 UTC and re-verifies all three with
the unchanged official verifiers. **Almost all of it is other people's work; every file and every improvement is
credited below, with the author, the change and a link.** Questions, reviews and corrections are welcome in the
comments.

| Task | File | Built by (latest change first) |
|---|---|---|
| invsqrt | `94b29cc5` | WCao24's 44-CSWAP normaliser on maoxin123's `a15b0d73` |
| DQI | Atakan Aldemir's iso-gap h5 file | Atakan's four-Toffoli GF(64) inverse character, on our readout-window transplant (version 1 of this notebook) of Atakan's `bb4287e7` |
| chemistry | npz `f21b8a87` + 15 loaders | prvsiyan's constrained refit of the weights, on maoxin123's v22 loaders (4,097 Toffolis per pass) and table `8afd4c67` |

## Acknowledgements
Thank you to everyone whose public work this notebook stands on. In particular:
**Alex Chilton**, whose *Verified Three-Task Portfolio* set the standard for assembling, pinning and re-verifying the
public frontier, and whose credit tables we follow; **Atakan Aldemir** for the DQI file used here and the readout
windows, shared products and GF(64) characters behind it; **WCao24** for the invsqrt normaliser; **prvsiyan** for the
chemistry refit and the careful written explanations; **maoxin123** for the invsqrt base and the chemistry loaders;
**manwithacat (James Barlow)** for the HW-25 join and the public circuits dataset we used to find the readout windows;
**Ertuğrul Özer (ozertuu)** for the pinned cache and table; **Heliosli**, **Kirill Tymchishin**, **Syed Asad Ali**,
**SomeJoint (Agniv Banerjee)**, **busyaprime (Akmal Xodarev)**, **yoyobar**, **mrmeows (team selenium)**, **Hosen42**,
**Gengsr**, **Ahmed Berat Özer**, **Lera (marculera)**, **Isabe11aM00re**, **Dmitrii Gluzdov**, **Leo Tang**,
**milos**, the team **i want to run gemma 9b**, **jday96314** (the decoder design) and **Oleg Sher (sheroleg)**, whose
explanatory notebook helped us read the invsqrt circuit. And Google Quantum AI and Kaggle for the competition, the
open kickmix simulator and the verifiers. If we missed you or misattributed a change, please tell us in the comments
and we will fix it in the next version.

**What is ours in this version.** The combination and its verification, and the readout-window transplant inside
the DQI (version 1 of this notebook: four controlled-swap readout windows of the `e9f54be3` line moved onto
`bb4287e7`, -48 Toffolis per shot), which Atakan Aldemir's file builds on.
New in this version, by us: a cross-lineage scan of 50 public DQI files found one more region (16777, `unconvert`) where
Syed Asad Ali's `3ec03d5d` is one Toffoli cheaper; the code below transplants it onto Atakan's file and keeps it
only if the official verifier shows zero phase errors on two salts and fewer Toffolis (printed below).

In [ ]:
import sys, os, pathlib, zipfile, json, time, subprocess, hashlib, shutil, math, difflib, re
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
COMP = pathlib.Path("/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation")
KM = COMP / "kickmix"; VER = COMP / "verifiers"; sys.path.insert(0, str(KM))
import kickmix as km
inp = pathlib.Path("/kaggle/input"); SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
sha = lambda b: hashlib.sha256(b).hexdigest()
def members(zpath):
    with zipfile.ZipFile(zpath) as z: return {n: z.read(n) for n in z.namelist()}
zips = {str(p): p for p in inp.rglob("submission.zip")}
wcao = members(next(p for k, p in zips.items() if "deterministic-open-portfolio" in k))
atak = members(next(p for k, p in zips.items() if "iso-gap-dqi-h5" in k))
inv = wcao["invsqrt.kmb"]; assert sha(inv).startswith("94b29cc5"), sha(inv)
npz = wcao["dfthcsos/submission.npz"]; assert sha(npz).startswith("f21b8a87"), sha(npz)
dqi = atak["dqi.kmb"]
log("invsqrt", sha(inv)[:12], "dqi (Atakan h5)", sha(dqi)[:12], "chem npz", sha(npz)[:12])
(SUB / "invsqrt.kmb").write_bytes(inv); (SUB / "dfthcsos" / "submission.npz").write_bytes(npz)
for n, b in wcao.items():
    if n.startswith("dfthcsos/angle_k_"): (SUB / n).write_bytes(b)
(SUB / "dqi.kmb").write_bytes(dqi)

In [ ]:
# Our addition (kept only if the official verifier agrees): region 16777 ("unconvert") of Syed Asad Ali's
# `3ec03d5d` is one Toffoli cheaper than the same region of `cd71c482`. Where Atakan Aldemir's file still holds the
# `cd71c482` bytes of that region, swap in Syed's version and keep it only if two salts show zero phase errors and fewer
# Toffolis than Atakan's file.
TRANSPLANT = {16777: "3ec03d5d3ae3"}
src = {p.stem[:12]: p for p in inp.rglob("*.kmb") if p.stem[:12] in ("cd71c4822aa0", "3ec03d5d3ae3")}
TMP = pathlib.Path("/tmp/t"); TMP.mkdir(exist_ok=True)
for k, p in src.items(): km.Circuit.from_file(p, format="kmb").to_file(TMP / f"{k}.kmx", format="kmx")
km.Circuit.from_file(SUB / "dqi.kmb", format="kmb").to_file(TMP / "base.kmx", format="kmx")
def scan(path, keep):
    tags, kept, idx = [""], {}, 0
    for line in open(path):
        if line.startswith(("APPEND_TO_REGISTER", "REGISTER")): continue
        if line.startswith("INFO_TAG"): idx += 1; tags.append(line)
        if idx in keep: kept.setdefault(idx, []).append(line)
    return tags, kept
tA, A = scan(TMP / "cd71c4822aa0.kmx", set(TRANSPLANT)); tB, _ = scan(TMP / "base.kmx", set())
sm = difflib.SequenceMatcher(a=tA, b=tB, autojunk=False)
amap = {b.a + k: b.b + k for b in sm.get_matching_blocks() for k in range(b.size)}
jmap = {amap[i]: i for i in TRANSPLANT if i in amap}
_, Bk = scan(TMP / "base.kmx", set(jmap))
new = {}
for j, i in jmap.items():
    _, Hk = scan(TMP / f"{TRANSPLANT[i]}.kmx", {i})
    if Bk.get(j) == A.get(i): new[j] = Hk[i]; log(f"region {i}: base holds the cd71 bytes, transplant {TRANSPLANT[i][:8]} ({len(Hk[i])} lines)")
    else: log(f"region {i}: base differs from cd71, skipped")
TRANSPLANT_KEPT = False
if new:
    idx, done = 0, set()
    with open(TMP / "cand.kmx", "w") as f:
        for line in open(TMP / "base.kmx"):
            if line.startswith("INFO_TAG"): idx += 1
            if idx in new and not line.startswith(("APPEND_TO_REGISTER", "REGISTER")):
                if idx not in done: f.writelines(new[idx]); done.add(idx)
                continue
            f.write(line)
    km.Circuit.from_file(TMP / "cand.kmx", format="kmx").to_file(TMP / "cand.kmb", format="kmb")
    def vdqi(path, salt):
        env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
        r = subprocess.run([sys.executable, str(VER / "dqi/verify_submission_dqi.py"), str(path), "--salt", salt], capture_output=True, text=True, env=env, cwd=str(VER))
        return json.loads(r.stdout[r.stdout.index("{"):])
    c1, c2, b1 = vdqi(TMP / "cand.kmb", ""), vdqi(TMP / "cand.kmb", "s1"), vdqi(SUB / "dqi.kmb", "s1")
    for nm, r_ in (("candidate default", c1), ("candidate s1", c2), ("Atakan h5 s1", b1)):
        log(nm, {k: r_.get(k) for k in ("toffoli_count", "value_errors", "phase_errors")})
    if c1["phase_errors"] == 0 and c2["phase_errors"] == 0 and c2["toffoli_count"] < b1["toffoli_count"] - 0.5:
        shutil.copy(TMP / "cand.kmb", SUB / "dqi.kmb"); TRANSPLANT_KEPT = True
log("transplant kept:", TRANSPLANT_KEPT)

## Official verification of the exact bytes that go into `submission.zip`

In [ ]:
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"])
log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors", "repetition_factor")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
dqi_zero = res["dqi"]["score"] / res["dqi"]["repetition_factor"] * 1.1912604648115832
for name, dq in (("this run's DQI draw", res["dqi"]["score"]), ("DQI at zero failed shots", dqi_zero)):
    r = {"invsqrt": REF["invsqrt"] / res["invsqrt"]["score"], "dqi": REF["dqi"] / dq, "dfthcsos": REF["dfthcsos"] / res["dfthcsos"]["score"]}
    print(f"{name:26s}", {k: round(x, 6) for k, x in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 11, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res},
          open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written")

## Credits (who did what), with links
**invsqrt** `94b29cc5`
- WCao24: the 44-CSWAP input normaliser, -2.0 Toffolis
  ([VQCC Deterministic Open Portfolio](https://www.kaggle.com/code/wcao24/vqcc-deterministic-open-portfolio)).
- maoxin123: `a15b0d73`, Block A shift-and-add stages re-ordered, controlled-SWAP rewrites
  ([LB 25.303 | Combined invsqrt + refit chem](https://www.kaggle.com/code/maoxin123/lb-25-303-combined-invsqrt-refit-chem)).
- manwithacat (James Barlow): HW-25 join and the public circuits dataset
  ([VQCC Hedge Wizardry](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook)); Gengsr: the 46-CSWAP shifter
  ([LB 25.2268](https://www.kaggle.com/code/gengsr/lb-25-2268-shifter-46-own-chemistry-loaders)).
- The 62-qubit lineage: Syed Asad Ali (base circuit), Hosen42 (q25 CCX merge, measure-first rewrites), Kirill
  Tymchishin (carry port), Atakan Aldemir (CSWAP deletion), Ertuğrul Özer (`13b5bf7a` composition), busyaprime (Akmal
  Xodarev), yoyobar (19-bit conditional phase fusion), mrmeows / team selenium (the 63-qubit ancestor), Leo Tang and
  Ehan Mallick, as credited in their notebooks.

**DQI** (Atakan Aldemir's iso-gap h5 file)
- Atakan Aldemir: four-Toffoli GF(64) inverse character at 28 sites, -27.6 Toffolis per shot
  ([Quantum Circuits: Iso-Gap + DQI h5](https://www.kaggle.com/code/atakanaldemir/quantum-circuits-iso-gap-dqi-h5));
  earlier the five-Toffoli version and `bb4287e7`
  ([Shared Products and Controlled Refits](https://www.kaggle.com/code/atakanaldemir/qcc-shared-products-and-controlled-refits)),
  and the controlled-swap readout windows of `e9f54be3`
  ([Terminal Readout and Phase Folds](https://www.kaggle.com/code/atakanaldemir/qcc-terminal-readout-and-phase-folds)).
- asteriospanoras (this notebook, version 1): the readout windows transplanted onto `bb4287e7`, -48 Toffolis per shot.
- Team *i want to run gemma 9b*: `cd71c482` and `b4dc000c` (joint zero-chart rewrites, conditional reverse-phase
  schedule, prefix fusion), shipped publicly via Oleg Sher's notebook.
- Heliosli (`9fcda0c1`: retained products; the regions we transplanted were taken from this file), Kirill Tymchishin
  (measurement-order identities, retained-product audit), Syed Asad Ali (decoder `1ab3ac53`, measurement peephole),
  SomeJoint (narrow return), Ali Haydar Özdağ (flag-carry idea), Isabe11aM00re, MAIDANG, busyaprime and yoyobar (product
  caching), Leo Tang, mrmeows, manwithacat (release-and-recompute), daifanhao, Khattar and Gidney (2024, the paper the
  decoder follows), and jday96314 (the decoder design; its repository states no licence, so it is credited, not
  relicensed).

**Chemistry**
- prvsiyan: constrained refit of the weights `f21b8a87`
  ([Verifier-Guided Quantum Circuit Optimization](https://www.kaggle.com/code/prvsiyan/verifier-guided-quantum-circuit-optimization)).
- maoxin123: v22 loaders, 4,097 Toffolis per pass
  ([QCC v22 | 4097-Toffoli chemistry loaders](https://www.kaggle.com/code/maoxin123/qcc-v22-4097-toffoli-chemistry-loaders)).
- ozertuu (Ertuğrul Özer): table `8afd4c67` and the cache dataset. Syed Asad Ali: the 8 x 143 x 44 table and the
  loader compiler. Lera (marculera): the DF-THC pipeline and the C44 build. Ahmed Berat Özer: progressive rounding and
  group compression. Atakan Aldemir: the pi-gauge. Weight refits before prvsiyan's: Isabe11aM00re, maoxin123 (with
  manwithacat's `dfthc_refit`), Heliosli, busyaprime. Dmitrii Gluzdov, milos and Agniv Banerjee: earlier shapes.

**Assembly and verification practice**: Alex Chilton's [Verified Three-Task Portfolio](https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio)
(pinning by SHA-256, re-verifying with the stock verifiers, pricing a DQI file at its board draw), and WCao24's
deterministic, byte-reproducible build. **Analysis**: Oleg Sher's [Quantum Circuits from Zero](https://www.kaggle.com/code/sheroleg/quantum-circuits-from-zero-1-sqrt-x-explained).
Our companion analysis notebook: [QCC Toolkit: Simulator & Cost Anatomy](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-what-works-what-fails-and-why).
Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each file keeps its authors' terms; nothing here is
relicensed. The code of this notebook is Apache-2.0.